In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: LBCO, HRPT

This minimalistic example is designed to show how Rietveld refinement
can be performed when both the crystal structure and experiment are
defined directly in code. Only the experimentally measured data is
loaded from an external file. It also shows how to switch calculation
engine.

For this example, constant-wavelength neutron powder diffraction data
for La0.5Ba0.5CoO3 from HRPT at PSI is used.

It does not contain any advanced features or options, and includes no
comments or explanations — these can be found in the other tutorials.
Default values are used for all parameters if not specified. Only
essential and self-explanatory code is provided.

The example is intended for users who are already familiar with the
EasyDiffraction library and want to quickly get started with a simple
refinement. It is also useful for those who want to see what a
refinement might look like in code. For a more detailed explanation of
the code, please refer to the other tutorials.

## 🛠️ Import Library

In [2]:
import easydiffraction as edi

## 📦 Define Project

In [3]:
project = edi.Project(name='lbco_hrpt')

## 🧩 Define Structure

In [4]:
project.structures.create(name='lbco')

In [5]:
structure = project.structures['lbco']

In [6]:
structure.space_group.name_h_m = 'P m -3 m'
structure.space_group.coord_system_code = '1'

In [7]:
structure.cell.length_a = 3.88

In [8]:
structure.atom_sites.create(
    id='La',
    type_symbol='La',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    adp_iso=0.5,
    occupancy=0.5,
)
structure.atom_sites.create(
    id='Ba',
    type_symbol='Ba',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    adp_iso=0.5,
    occupancy=0.5,
)
structure.atom_sites.create(
    id='Co',
    type_symbol='Co',
    fract_x=0.5,
    fract_y=0.5,
    fract_z=0.5,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='O',
    type_symbol='O',
    fract_x=0,
    fract_y=0.5,
    fract_z=0.5,
    adp_iso=0.5,
)

In [9]:
project.display.structure(struct_name='lbco')

Structure 🧩 'lbco' (Atom view type: 'covalent')


## 🔬 Define Experiment

In [10]:
data_path = edi.download_data('meas-lbco-hrpt', destination='data')

Getting data...


Data 'meas-lbco-hrpt': La0.5Ba0.5CoO3, HRPT (PSI), 300 K


✅ Data 'meas-lbco-hrpt' downloaded to '../../../data/meas-lbco-hrpt.xye'


In [11]:
project.experiments.add_from_data_path(
    name='hrpt',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
)

Data loaded successfully


Experiment 🔬 'hrpt'. Number of data points: 3098.


In [12]:
experiment = project.experiments['hrpt']

In [13]:
experiment.instrument.setup_wavelength = 1.494
experiment.instrument.calib_twotheta_offset = 0.6

In [14]:
experiment.peak.broad_gauss_u = 0.1
experiment.peak.broad_gauss_v = -0.1
experiment.peak.broad_gauss_w = 0.1
experiment.peak.broad_lorentz_y = 0.1

In [15]:
experiment.excluded_regions.create(id='1', start=0, end=5)
experiment.excluded_regions.create(id='2', start=165, end=180)

In [16]:
experiment.background.auto_estimate()

In [17]:
experiment.linked_structures.create(structure_id='lbco', scale=10.0)

## 🚀 Perform Analysis

### Without Constraints

In [18]:
structure.cell.length_a.free = True

structure.atom_sites['La'].adp_iso.free = True
structure.atom_sites['Ba'].adp_iso.free = True
structure.atom_sites['Co'].adp_iso.free = True
structure.atom_sites['O'].adp_iso.free = True

In [19]:
experiment.instrument.calib_twotheta_offset.free = True

experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.peak.broad_gauss_w.free = True
experiment.peak.broad_lorentz_y.free = True

for point in experiment.background:
    point.intensity.free = True

experiment.linked_structures['lbco'].scale.free = True

In [20]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.31,164.50,
2,28,1.39,31.70,80.7% ↓
3,45,2.06,10.78,66.0% ↓
4,63,2.97,7.96,26.2% ↓
5,80,3.64,3.06,61.6% ↓
6,98,4.37,2.04,33.1% ↓
7,115,5.11,1.29,36.6% ↓
8,133,5.91,1.27,2.2% ↓
9,240,10.73,1.27,


🏆 Best goodness-of-fit (reduced χ²) is 1.27 at iteration 239


✅ Fitting complete.


In [21]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),10.73
4,🔁 Iterations,237
5,📏 Goodness-of-fit (reduced χ²),1.27
6,"📏 R-factor (Rf, %)",5.60
7,"📏 R-factor squared (Rf², %)",5.23
8,"📏 Weighted R-factor (wR, %)",4.40


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lbco,cell,,length_a,Å,3.8800,3.8909,0.0000,0.28 % ↑
2,lbco,atom_site,La,adp_iso,Å²,0.5000,0.4871,130.9321,2.59 % ↓
3,lbco,atom_site,Ba,adp_iso,Å²,0.5000,0.4872,212.9950,2.57 % ↓
4,lbco,atom_site,Co,adp_iso,Å²,0.5000,0.2619,0.0613,47.61 % ↓
5,lbco,atom_site,O,adp_iso,Å²,0.5000,1.4090,0.0167,181.80 % ↑
6,hrpt,linked_structure,lbco,scale,,10.0000,9.1342,0.0633,8.66 % ↓
7,hrpt,peak,,broad_gauss_u,deg²,0.1000,0.0812,0.0031,18.80 % ↓
8,hrpt,peak,,broad_gauss_v,deg²,-0.1000,-0.1146,0.0066,14.62 % ↑
9,hrpt,peak,,broad_gauss_w,deg²,0.1000,0.1205,0.0032,20.52 % ↑
10,hrpt,peak,,broad_lorentz_y,deg,0.1000,0.0833,0.0021,16.66 % ↓


In [22]:
project.display.fit.correlations()

In [23]:
project.display.pattern(expt_name='hrpt')

### With Constraints

In [24]:
# As can be seen from the parameter-correlation plot, the isotropic
# displacement parameters of La and Ba are highly correlated. Because
# La and Ba share the same mixed-occupancy site, their contributions to
# the neutron diffraction pattern are difficult to separate, especially
# since their coherent scattering lengths are not very different.
# Therefore, it is necessary to constrain them to be equal. First we
# define aliases and then use them to create a constraint.
project.analysis.aliases.create(
    id='biso_La',
    param=project.structures['lbco'].atom_sites['La'].adp_iso,
)
project.analysis.aliases.create(
    id='biso_Ba',
    param=project.structures['lbco'].atom_sites['Ba'].adp_iso,
)
project.analysis.constraints.create(expression='biso_Ba = biso_La')

In [25]:
project.analysis.minimizer.show_supported()
project.analysis.minimizer.type = 'lmfit'

Minimizer types


,,Type,Description
1,,bumps,BUMPS library using the default Levenberg-Marquardt method
2,,bumps (amoeba),BUMPS library with Nelder-Mead simplex method
3,,bumps (de),BUMPS library with differential evolution method
4,,bumps (dream),BUMPS library with DREAM Bayesian sampling
5,,bumps (lm),BUMPS library with Levenberg-Marquardt method
6,,dfols,DFO-LS library for derivative-free least-squares optimization
7,,emcee,emcee affine-invariant ensemble Bayesian sampling
8,,lmfit,LMFIT library using the default Levenberg-Marquardt method
9,,lmfit (least_squares),LMFIT library with SciPy's trust region reflective algorithm
10,*,lmfit (leastsq),LMFIT library with Levenberg-Marquardt least squares method


Current minimizer changed to


lmfit


In [26]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.05,1.27,
2,20,0.95,1.27,


🏆 Best goodness-of-fit (reduced χ²) is 1.27 at iteration 19


✅ Fitting complete.


In [27]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),0.95
4,🔁 Iterations,17
5,📏 Goodness-of-fit (reduced χ²),1.27
6,"📏 R-factor (Rf, %)",5.60
7,"📏 R-factor squared (Rf², %)",5.23
8,"📏 Weighted R-factor (wR, %)",4.40


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lbco,cell,,length_a,Å,3.8909,3.8909,0.0000,0.00 % ↓
2,lbco,atom_site,La,adp_iso,Å²,0.4871,0.4871,0.0275,0.01 % ↑
3,lbco,atom_site,Co,adp_iso,Å²,0.2619,0.2619,0.0568,0.00 % ↓
4,lbco,atom_site,O,adp_iso,Å²,1.4090,1.4090,0.0160,0.00 % ↓
5,hrpt,linked_structure,lbco,scale,,9.1342,9.1342,0.0532,0.00 % ↑
6,hrpt,peak,,broad_gauss_u,deg²,0.0812,0.0812,0.0031,0.00 % ↓
7,hrpt,peak,,broad_gauss_v,deg²,-0.1146,-0.1146,0.0066,0.00 % ↓
8,hrpt,peak,,broad_gauss_w,deg²,0.1205,0.1205,0.0032,0.00 % ↓
9,hrpt,peak,,broad_lorentz_y,deg,0.0833,0.0833,0.0021,0.00 % ↑
10,hrpt,instrument,,twotheta_offset,deg,0.6225,0.6225,0.0010,0.00 % ↓


In [28]:
project.display.fit.correlations()

In [29]:
project.display.pattern(expt_name='hrpt')

### Switch Calculator

In [30]:
experiment.calculator.show_supported()

Calculator types


,,Type,Description
1,,crysfml,CrysFML library for crystallographic calculations
2,*,cryspy,CrysPy library for crystallographic calculations


In [31]:
experiment.calculator.type = 'crysfml'

Calculator for experiment 'hrpt' changed to


crysfml


In [32]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.04,1.27,
2,60,2.26,1.26,


🏆 Best goodness-of-fit (reduced χ²) is 1.26 at iteration 40


✅ Fitting complete.


In [33]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),2.26
4,🔁 Iterations,57
5,📏 Goodness-of-fit (reduced χ²),1.26
6,"📏 R-factor (Rf, %)",5.57
7,"📏 R-factor squared (Rf², %)",5.22
8,"📏 Weighted R-factor (wR, %)",4.39


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lbco,cell,,length_a,Å,3.8909,3.8908,0.0000,0.00 % ↓
2,lbco,atom_site,La,adp_iso,Å²,0.4871,0.4765,0.0227,2.17 % ↓
3,lbco,atom_site,Co,adp_iso,Å²,0.2619,0.2873,0.0481,9.68 % ↑
4,lbco,atom_site,O,adp_iso,Å²,1.4090,1.4120,0.0155,0.21 % ↑
5,hrpt,linked_structure,lbco,scale,,9.1342,9.1280,0.0508,0.07 % ↓
6,hrpt,peak,,broad_gauss_u,deg²,0.0812,0.0837,0.0020,3.06 % ↑
7,hrpt,peak,,broad_gauss_v,deg²,-0.1146,-0.1198,0.0037,4.55 % ↑
8,hrpt,peak,,broad_gauss_w,deg²,0.1205,0.1230,0.0027,2.03 % ↑
9,hrpt,peak,,broad_lorentz_y,deg,0.0833,0.0827,0.0020,0.74 % ↓
10,hrpt,instrument,,twotheta_offset,deg,0.6225,0.6199,0.0009,0.41 % ↓


In [34]:
project.display.fit.correlations()

In [35]:
project.display.pattern(expt_name='hrpt')

## 💾 Save Project

In [36]:
project.save_as(dir_path='projects/refine-lbco-hrpt-from-data')

Saving project 📦 'lbco_hrpt' to '../../../projects/refine-lbco-hrpt-from-data'


├── 📄 project.edi


├── 📁 structures/


│   └── 📄 lbco.edi


├── 📁 experiments/


│   └── 📄 hrpt.edi


├── 📁 analysis/


│   └── 📄 analysis.edi


└── 📁 reports/


    └── 📄 lbco_hrpt.html
